# TP5 - Functional Programming and scikit-learn Pipelines

## 🎯 Goal of this TP

In this TP, you will progressively transform a working ML script into a clean, functional, and reliable training pipeline.

You will learn:
- why functional programming matters in ML projects,
- how scikit-learn pipelines help enforce good practices,
- how to refactor procedural code into reusable functions.

This TP is progressive. You are not expected to finish everything. The goal is to understand the approach.

## Prerequisites

Before starting:
- you must have a working titanic.py (from previous TPs),
- the script must run from the terminal,
- dependencies must be installed in a virtual environment.

## 🎯 Pedagogical objective

The goal of this TP is to help you move:
- from a working ML script
- to a well-designed ML codebase that is readable, reliable, and reusable.

By the end of this TP, you should understand:
- why scikit-learn pipelines exist,
- how pipelines improve code reliability,
- how functional programming helps structure ML projects,
- how to organize a project for long-term maintainability.

## Part 1 – Why pipelines and functional programming?

Open your current titanic.py and observe:
- Is preprocessing mixed with training?
- Are transformations repeated?
- Is the execution order implicit?
- Would you feel confident modifying this script in 3 months?

**👉 This TP answers: "How do I design ML code that survives time?"**

### A common ML problem

A "naive" ML script often:
- mixes preprocessing, training, and evaluation,
- relies on execution order,
- duplicates logic,
- is hard to modify safely.

👉 Functions and pipelines are tools designed to solve these issues.

### Key ideas to remember

- One function = one responsibility
- Pipelines make transformations explicit and ordered
- Good ML code should be:
  - readable
  - testable
  - reproducible
  - safe by design

In [ ]:
import sys
import os
from pathlib import Path

# Get project root directory (parent of notebooks directory)
notebook_dir = Path().resolve()
project_root = notebook_dir.parent

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Change working directory to project root
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current working directory: {os.getcwd()}")

## Part 2 – Understanding the code without a scikit-learn pipeline

### Step 2.1 – Observe procedural preprocessing

First, let's load and split the data:

In [ ]:
from src.data.load import load_local_data
from src.data.split import split_data

# Load data
data = load_local_data("data/raw/titanic.csv")

# Split data into training and testing sets
X_train, X_test, y_train, y_test = split_data(
    data=data,
    target_column="Survived",
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

In [ ]:
data.columns

Look at the following code:

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import pandas as pd

# Model hyperparameters
n_trees = 20

# Variable definition
numeric_features = ["Age", "Fare"]
categorical_features = ["Embarked", "Sex"]

# PREPROCESSING
# Select only the features we need for training
all_features = numeric_features + categorical_features
X_train = X_train[all_features].copy()
X_test = X_test[all_features].copy()

# Handling missing values for numerical features
num_imputer = SimpleImputer(strategy="median")
X_train[numeric_features] = num_imputer.fit_transform(X_train[numeric_features])
X_test[numeric_features] = num_imputer.transform(X_test[numeric_features])

# Scaling numerical features
scaler = MinMaxScaler()
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# Handling missing values for categorical features
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train[categorical_features] = cat_imputer.fit_transform(X_train[categorical_features])
X_test[categorical_features] = cat_imputer.transform(X_test[categorical_features])

# One-hot encoding categorical features
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_encoded = encoder.fit_transform(X_train[categorical_features])
X_test_encoded = encoder.transform(X_test[categorical_features])

# Convert encoded features into a DataFrame
X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(categorical_features), index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(categorical_features), index=X_test.index)

# Drop original categorical columns and concatenate encoded ones
X_train = X_train.drop(columns=categorical_features).join(X_train_encoded)
X_test = X_test.drop(columns=categorical_features).join(X_test_encoded)

# MODEL TRAINING
# Defining the model
model = RandomForestClassifier(n_estimators=n_trees)
# Fitting the model
model.fit(X_train, y_train)

# EVALUATION
# Scoring
rdmf_score = model.score(X_test, y_test)
print(f"{rdmf_score:.1%} correct answers on test data for validation")

# Confusion matrix
print(20 * "-")
print("Confusion matrix")
print(confusion_matrix(y_test, model.predict(X_test)))

The code above performs exactly the same operations as a scikit-learn pipeline, but manually and step by step.

At first glance, it works. However, it illustrates why pipelines exist.

Let's break down what this code is doing and why it is fragile.

### 1️⃣ Preprocessing is split into many independent steps

The preprocessing phase is spread across multiple blocks:
- numerical imputation
- numerical scaling
- categorical imputation
- categorical encoding
- DataFrame reconstruction
- column dropping and concatenation

Each step:
- must be executed in the correct order,
- must be applied consistently to both train and test data,
- must reuse the same fitted transformers.

This creates implicit dependencies between code blocks.

### 2️⃣ Manual state management is error-prone

Each transformer (SimpleImputer, MinMaxScaler, OneHotEncoder) has an internal state learned on X_train.

You must manually ensure that:
- `.fit()` is called only on training data,
- `.transform()` is called on test data,
- the same objects are reused.

A single mistake (e.g. fitting again on test data) introduces data leakage. Pipelines prevent this by construction.

### 3️⃣ The code mutates data in place

Notice:
- `X_train[numeric_features] = ...`
- `X_train = X_train.drop(...).join(...)`

This means:
- X_train is progressively modified,
- the original data structure is lost,
- debugging becomes harder.

If an error occurs midway, it is difficult to recover the original state. Pipelines keep transformations pure and encapsulated.

### 4️⃣ Feature engineering logic is duplicated and tangled

The logic for categorical encoding requires:
- manual reconstruction of DataFrames,
- manual column name handling,
- index alignment.

This is:
- verbose,
- easy to break,
- hard to extend.

Adding one new categorical feature requires changes in multiple places.

With a pipeline:
- feature handling is declarative,
- column selection is centralized.

### 5️⃣ Preprocessing and modeling are loosely coupled

Preprocessing and modeling are defined separately, but their compatibility is implicit.

If you:
- change the preprocessing,
- add a new feature,
- change the model,

you must ensure by yourself that everything still matches.

A pipeline makes preprocessing + model a single, consistent object.

## Part 3 – Introducing scikit-learn pipelines

This code below defines a complete machine-learning pipeline that includes:
- data loading
- preprocessing (numerical + categorical)
- model definition
- training

All steps are combined into a single object (pipe) to ensure consistency and reliability.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

n_trees = 20
MAX_DEPTH = None
MAX_FEATURES = "sqrt"

numeric_features = ["Age", "Fare"]
categorical_features = ["Embarked", "Sex"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=MAX_DEPTH,
            max_features=MAX_FEATURES
        )),
    ]
)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = split_data(
    data=data,
    target_column="Survived",
    test_size=0.2,
    random_state=42
)

pipe.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix

# MODEL TRAINING
# Training the pipeline (this fits all preprocessing steps and the classifier)
pipe.fit(X_train, y_train)

# EVALUATION
# Scoring
pipe_score = pipe.score(X_test, y_test)
print(f"{pipe_score:.1%} correct answers on test data for validation")

# Confusion matrix
print(20 * "-")
print("Confusion matrix")
print(confusion_matrix(y_test, pipe.predict(X_test)))

### 1️⃣ Imports

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
```

These imports cover:
- pandas: data manipulation
- SimpleImputer: handling missing values
- MinMaxScaler: scaling numerical variables
- OneHotEncoder: encoding categorical variables
- Pipeline: chaining transformations
- ColumnTransformer: applying different transformations to different columns
- RandomForestClassifier: the ML model

### 2️⃣ Data loading and split

```python
# Load and split data
X_train, X_test, y_train, y_test = split_data(
    data=data,
    target_column="Survived",
    test_size=0.2,
    random_state=42
)
```

Note: The `data` variable should be loaded in a previous cell using `load_local_data()`.

- X_train / X_test : input features
- y_train / y_test : target variable (Survived)
- The model will learn from X_train → y_train

⚠ Important concept: The pipeline does not need to know which column is the target. That separation is done outside the pipeline using the `split_data` function.

### 3️⃣ Model hyperparameters

```python
n_trees = 20
MAX_DEPTH = None
MAX_FEATURES = "sqrt"
```

These parameters control the Random Forest:
- n_trees : number of trees
- max_depth : maximum depth of trees
- max_features : number of features considered at each split

Later, these parameters can be externalized (CLI arguments, config files).

### 4️⃣ Feature selection

```python
numeric_features = ["Age", "Fare"]
categorical_features = ["Embarked", "Sex"]
```

Here we explicitly declare:
- which features are numerical
- which features are categorical

This is crucial for:
- readability
- reproducibility
- avoiding silent errors

### 5️⃣ Numerical preprocessing pipeline

```python
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
    ]
)
```

This pipeline:
- replaces missing numerical values with the median
- scales values between 0 and 1

Key idea:
- transformations are ordered
- learned only on training data
- reused automatically on test data

### 6️⃣ Categorical preprocessing pipeline

```python
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
```

This pipeline:
- fills missing values with the most frequent category
- converts categories into numerical features using one-hot encoding

Again:
- all logic is grouped
- order is guaranteed
- no manual DataFrame manipulation is required

### 7️⃣ ColumnTransformer: applying the right pipeline to the right columns

```python
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)
```

ColumnTransformer:
- applies numeric_transformer only to numerical columns
- applies categorical_transformer only to categorical columns
- merges the results automatically

This avoids:
- duplicated code
- column mismatch errors
- manual joins

### 8️⃣ Full pipeline (preprocessing + model)

```python
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=MAX_DEPTH,
            max_features=MAX_FEATURES
        )),
    ]
)
```

This pipeline:
- preprocesses the data
- trains the classifier

Preprocessing and modeling are now:
- inseparable
- consistent
- reusable

This is a production-ready design.

### 9️⃣ Training the pipeline

```python
pipe.fit(X_train, y_train)
```

This single line:
- fits all preprocessing steps on training data
- fits the Random Forest model
- stores everything inside pipe

After this:
- `pipe.predict(X_test)` works directly
- no need to reapply preprocessing manually

### Step 3.3 – Inspect the pipeline

In a new cell, simply write:

In [ ]:
pipe

Observe:
- the ordered steps,
- how preprocessing and modeling are connected.

✍ Question: Is the modeling workflow clearer than in the procedural version? Why?

## Part 4 – Understanding pipeline internals

### Step 4.1 – Access pipeline steps

Try the following in the notebook:

In [ ]:
pipe.named_steps

Questions:
- What keys do you see?
- What object corresponds to preprocessing?
- What object corresponds to the classifier?

### Step 4.2 – Access only preprocessing

Run:

In [ ]:
pipe[:-1]

This removes the classifier step.

✍ Question: What does this object represent conceptually?

### Step 4.3 – Apply preprocessing to new data

Create new data:

In [ ]:
import numpy as np

new_data = pd.DataFrame({
    "Age": [22, np.nan, 35, 28, np.nan],
    "Fare": [7.25, 8.05, np.nan, 13.00, 15.50],
    "Embarked": ["S", "C", np.nan, "Q", "S"],
    "Sex": ["male", "female", "male", np.nan, "female"]
})

Apply preprocessing only:

In [ ]:
X_new_preprocessed = pipe[:-1].transform(new_data)
pd.DataFrame(X_new_preprocessed)

✍ Discussion:
- How many lines were required?
- Would this be easy without pipelines?

## Part 5 – Functional refactoring of the script

Now return to titanic.py.

### 5.1 Create a pipeline builder function

At the top of the file, add:

In [ ]:
def build_pipeline(
    n_trees: int = 20,
    numeric_features: list = ["Age", "Fare"],
    categorical_features: list = ["Embarked", "Sex"],
):
    """
    Build and return a scikit-learn pipeline.
    """
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(
                n_estimators=n_trees,
            )),
        ]
    )
    return pipe

In [ ]:
pipe = build_pipeline(n_trees=20)
pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)

👉 Move all pipeline-related code inside this function.

⚠ Do not call `.fit()` inside the function.

### 5.2 Create an evaluation function

Add:

In [ ]:
from sklearn.metrics import confusion_matrix

def evaluate_model(model, X_test, y_test):
    score = model.score(X_test, y_test)
    cm = confusion_matrix(y_test, model.predict(X_test))
    return score, cm

In [ ]:
evaluate_model(pipe, X_test, y_test)

### 5.3 Simplify the main script logic

Your script should now look like:

In [ ]:
if __name__ == "__main__":
    # load data
    # build pipeline
    # fit model
    # evaluate model

Remove:
- duplicated code,
- intermediate experiments,
- unused variables ("zombie code").

Run the script once to ensure it still works.

## Part 6 – Project structure

### Target minimal structure

Reorganize your project as follows:

```
project/
├── data/
│   └── raw/
│       ├── titanic.csv
│   └── processed_data
|       ...
├── src/
│   └── train.py
├── README.md
└── .gitignore
```

👉 Move your script into `src/train.py`

👉 Update data paths accordingly

(Optional: split pipeline and evaluation into separate files)